## Stability of RK4 for the spectral heat equation

**Rule of thumb** (Trefethen, *Spectral Methods in MATLAB*, Ch. 10, p. 103): the method of lines
is stable if the eigenvalues of the (linearised) spatial discretisation operator, scaled by Δt,
lie in the stability region of the time-stepping formula.

**Semi-discretisation.** In Fourier space, u_t = ν u_xx becomes the decoupled system
dû_k/dt = λ_k û_k with λ_k = −ν k². The operator is diagonalised by the DFT, which is unitary,
so here the rule of thumb is exact rather than heuristic.

**One RK4 step.** For y' = λy, one RK4 step gives y_{n+1} = R(z) y_n with z = λh and
R(z) = 1 + z + z²/2 + z³/6 + z⁴/24. Hence y_n = R(z)^n y_0, and the scheme is stable
if and only if |R(λ_k h)| ≤ 1 for every k.

**The limit.** All λ_k lie on the negative real axis. There |R(z)| ≤ 1 for −2.7853 ≤ z ≤ 0;
the left endpoint is the real root of z³ + 4z² + 12z + 24 = 0, where R = +1. The most negative
eigenvalue belongs to the Nyquist mode k = N/2 (kept, since the second derivative is even in k), so

  h ≤ h_crit = 2.7853 / (ν (N/2)²) ≈ 0.1088   for N = 32, ν = 0.1.

**Prediction.** For h > h_crit, R > 1 (not R < −1), so the Nyquist mode grows monotonically,
by a factor R per step, from a roundoff seed ε ~ 10⁻¹⁶: amplitude ≈ ε R(z)^n after n steps.
Near h_crit, R ≈ 1, so growth is slow and a finite run can look stable when it is not.

In [ ]:
import numpy as np
from dmhd.solvers.heat1d import solve_heat
from dmhd.spectral import wavenumbers
from dmhd.timestep import rk4_step

T = 40
v = 0.1
N = 32
steps = [350, 360, 370, 380]
dx = 2*np.pi/N
x = dx*np.arange(N)
initial = np.sin(x) + 0.5*np.sin(3*x)
analytic = np.exp(-v*T)*np.sin(x) + 0.5*np.exp(-9*v*T)*np.sin(3*x)
err = dict()
for n_steps in steps:
    sol = solve_heat(initial, v, T, n_steps)
    err[T/n_steps] = np.max(np.abs(sol-analytic))
print(err)


### Findings: blow-up runs (T = 40, N = 32, ν = 0.1)

| n_steps | h | z = λ_{N/2} h | R(z) | R(z)ⁿ | max error at T |
|---|---|---|---|---|---|
| 350 | 0.1143 | −2.926 | 1.233 | 7.3×10³¹ | ≈5×10¹⁵ |
| 360 | 0.1111 | −2.844 | 1.093 | 7.8×10¹³ | ≈5×10⁻³ |
| 370 | 0.1081 | −2.768 | 0.974 | 5.0×10⁻⁵ | 8.4×10⁻¹² |
| 380 | 0.1053 | −2.695 | 0.872 | 2.3×10⁻²³ | 7.6×10⁻¹² |

- **The blow-up is the predicted one.** Both unstable runs give the same roundoff seed, ε = error / Rⁿ ≈ 7×10⁻¹⁷, so the error is machine roundoff in the Nyquist mode amplified by R per step. (The exact blow-up values differ slightly between machines, e.g. 5.0×10¹⁵ on macOS and 5.2×10¹⁵ on Linux, because the roundoff seed does; ε ≈ 7×10⁻¹⁷ on both.)
- **Near the limit, a finite run can look stable.** h = 0.1111 is above h_crit = 0.1088, but R = 1.09 is so close to 1 that the error only reaches 5×10⁻³ by T = 40. "Bounded at time T" does not prove stability near h_crit.
- **The stable runs** (8×10⁻¹² error) show ordinary RK4 time-stepping error in the physical modes; the Nyquist mode decays by Rⁿ ≤ 5×10⁻⁵.
- **Per-step growth (cells below).** One RK4 step multiplies the Nyquist mode by exactly R(λ_{N/2} h): measured to 7×10⁻¹⁶ across the sweep, and the growth at h_crit (exact root of the cubic) is 1 to ~10⁻¹⁴.

**Conclusion:** the stability limit h_crit = 2.7853 / (ν (N/2)²) is confirmed both by the per-step growth factor and by the blow-up runs. Because h_crit ∝ 1/(ν N²), doubling the resolution forces 4× more steps; this stiffness is what integrating-factor RK4 removes.


In [ ]:
initial_2 = (1e-10)*(-1)**np.arange(N)
steps = range(380, 350, -1)
growth = np.zeros(len(steps))
wavenums = wavenumbers(N)
def rhs(t, u_h):
    return -v*(wavenums**2)*u_h
def R(h, lam):
    return 1 + (h*lam) +  1/2*(h*lam)**2 + 1/6*(h*lam)**3 + 1/24*(h*lam)**4
for a, i in enumerate(steps):
    u_hat = np.fft.fft(initial_2)
    h = T/i
    u_hat_plusone = rk4_step(rhs, 0, u_hat, h)
    growth[a] = np.abs(u_hat_plusone[N//2]/u_hat[N//2])

print([R(T/i, -v*(N//2)**2) for i in steps])
print(growth)

for (i, step) in enumerate(steps):
    assert abs(R(T/step, -v*(N//2)**2) - growth[i]) < 1e-12

In [ ]:
import matplotlib.pyplot as plt

lam_max = -v * (N // 2) ** 2                       # most negative eigenvalue (Nyquist mode)
roots = np.roots([1, 4, 12, 24])                   # R(z) = 1 with z != 0
z_crit = roots[np.isreal(roots)].real[0]           # ≈ -2.7853
h_crit = z_crit / lam_max

def measured_growth(h):
    """|û_{N/2}| after one RK4 step divided by |û_{N/2}| before."""
    u_hat = np.fft.fft(initial_2)
    u_next = rk4_step(rhs, 0.0, u_hat, h)
    return np.abs(u_next[N // 2] / u_hat[N // 2])

hs_fine = np.linspace(0.0, 0.13, 400)              # theory curve
hs_meas = np.linspace(0.005, 0.13, 26)             # one RK4 step each
g_meas = np.array([measured_growth(h) for h in hs_meas])
g_crit = measured_growth(h_crit)

print(f"z_crit = {z_crit:.6f},  h_crit = {h_crit:.6f}")
print(f"measured growth at h_crit = {g_crit:.15f}  (|g - 1| = {abs(g_crit - 1):.1e})")
print(f"max |measured - R| over the sweep = {np.max(np.abs(g_meas - np.abs(R(hs_meas, lam_max)))):.1e}")

fig, ax = plt.subplots(figsize=(6.5, 4.2), constrained_layout=True)
ax.axhspan(0, 1, color="#2a78d6", alpha=0.06, lw=0)
ax.plot(hs_fine, np.abs(R(hs_fine, lam_max)), color="#2a78d6", lw=1.5,
        label=r"theory $|R(\lambda_{N/2} h)|$")
ax.plot(hs_meas, g_meas, "o", color="#eb6834", ms=5, label="measured, one RK4 step")
ax.plot([h_crit], [g_crit], "s", color="#0b0b0b", ms=7, zorder=5,
        label=rf"$h_{{crit}}={h_crit:.4f}$, growth $={g_crit:.12f}$")
ax.axhline(1, color="gray", ls="--", lw=1)
ax.axvline(h_crit, color="gray", ls=":", lw=1)
ax.text(0.004, 0.33, "stable: |R| < 1", color="#52514e", fontsize=8)
ax.text(h_crit + 0.002, 1.12, "unstable", color="#52514e", fontsize=8)
ax.set_xlabel("time step $h$")
ax.set_ylabel(r"per-step growth of the Nyquist mode $|\hat u^{n+1}_{N/2}/\hat u^{n}_{N/2}|$")
ax.set_title(rf"RK4 stability, heat equation ($N={N}$, $\nu={v}$)", fontsize=10)
ax.set_xlim(0, 0.13)
ax.set_ylim(0.2, 1.3)
ax.legend(loc="upper left", fontsize=8, frameon=False)
ax.grid(True, color="#e4e3df", lw=0.8)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.show()